# Phase 7 — Distribution Shift Detection and Robustness Evaluation

## 1. Objective
In real-world water distribution networks, environmental changes (e.g., summer vs. winter temperatures, seasonal demand shifts) and physical sensor drift (e.g., sensor recalibration, pipe degradation, pump operational changes) cause incoming sensor data distributions to deviate from the training distribution.

The goal of Phase 7 is to evaluate whether our trained anomaly detection models remain robust when incoming sensor data shifts, establish a statistical distribution shift detector, measure performance degradation under shift, and provide a reliable trigger for future adaptive learning (Phase 8).

### Core Pipeline Architecture
```
Training Data (Jan–Aug 2018)
      ↓
Reference Distribution (per-feature statistics & bootstrap threshold)
      ↓
Incoming Sensor Data (Test period Nov–Dec / Shifted conditions)
      ↓
Distribution Shift Detector (Kolmogorov-Smirnov two-sample test)
      ↓
[ STABLE / SHIFT DETECTED ]
      ↓
Anomaly Detection Model (SSL-pretrained Temporal Transformer)
      ↓
Performance Degradation & Robustness Evaluation
```

**Important Principles**:
- **Distribution Shift Detection** ≠ **Anomaly Detection**: Shift detection determines whether the *underlying data-generating distribution* has changed across a time window. Anomaly detection evaluates whether *a specific short sequence* indicates abnormal physical behavior (leakage, burst, pump fault).
- **Zero Data Leakage**: The reference distribution and shift threshold are computed *strictly on training data* without using test labels.


## 2. Existing Data Pipeline
We inspect the existing preprocessing pipeline, sensor features, sequence construction, and chronological train/validation/test split established in Phase 2.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, '../src')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from config import SENSOR_COLS, RESULTS_DIR, FIGURES_DIR, DATA_PROC, MODELS_DIR

DRIFT_RES = RESULTS_DIR / 'drift'
DRIFT_FIGS = FIGURES_DIR / 'drift'

# Load preprocessing metadata
with open(DATA_PROC / 'preprocessing_meta.json') as f:
    prep_meta = json.load(f)

print(f"Monitored Sensor Channels ({len(prep_meta['sensor_cols'])}): {prep_meta['sensor_cols']}")
print(f"Total Model Features ({len(prep_meta['feature_cols'])}): {prep_meta['feature_cols']}")
print(f"Train split: {prep_meta['train_rows']:,} rows ({prep_meta['train_anomaly_pct']:.2f}% anomaly)")
print(f"Val split:   {prep_meta['val_rows']:,} rows ({prep_meta['val_anomaly_pct']:.2f}% anomaly)")
print(f"Test split:  {prep_meta['test_rows']:,} rows ({prep_meta['test_anomaly_pct']:.2f}% anomaly)")


## 3. Reference Distribution
The reference distribution captures the empirical baseline statistics of every sensor feature using **training data only**. We compute:
- Mean and Standard Deviation
- Median, Minimum, and Maximum
- 5th, 25th, 75th, and 95th percentiles
- Missing value percentage (verified 0%)


In [ ]:
# Load and inspect reference distribution statistics
ref_df = pd.read_csv(DRIFT_RES / 'reference_distribution.csv')
print(f"Reference distribution established for {len(ref_df)} features.")
ref_df[['feature', 'mean', 'std', 'median', 'min', 'max', 'p25', 'p75', 'missing_pct']]


## 4. Distribution-Shift Methodology

### Selected Method: Two-Sample Kolmogorov-Smirnov (KS) Test
We select the two-sample **Kolmogorov-Smirnov (KS) test** as the primary statistical method:
$$\text{KS}(F_1, F_2) = \sup_x |F_1(x) - F_2(x)|$$
Where $F_1(x)$ and $F_2(x)$ are the empirical cumulative distribution functions of the reference and incoming sensor measurements.

### Justification:
1. **Univariate & Continuous**: Water SCADA readings (pressures, flows, tank level) are continuous real-valued signals.
2. **Non-parametric**: No Gaussian or stationarity assumptions are imposed.
3. **Bounded & Scale-Free**: The KS statistic is bounded in $[0, 1]$, allowing direct comparisons across diverse sensor channels.
4. **Interpretable**: Represents the maximum absolute difference between empirical CDFs.

### Aggregation:
The overall system shift score is the mean KS statistic across monitored sensor channels:
$$\text{Shift Score} = \frac{1}{M} \sum_{j=1}^{M} \text{KS}_j$$

### Threshold Derivation (No Data Leakage):
The detection threshold is derived via non-parametric bootstrap resampling on the **reference training data**:
- 500 pairs of independent random subsets (30% size) are sampled from training data.
- The pairwise KS statistic is computed for each pair.
- The **99th percentile** of the bootstrap distribution is set as the threshold ($	au = 0.009148$).
- Any incoming shift score exceeding $	au$ indicates a statistically significant departure from the training regime ($p < 0.01$).


In [ ]:
with open(DRIFT_RES / 'detector_config.json') as f:
    det_cfg = json.load(f)

print(f"Detection Method:       {det_cfg['method']}")
print(f"Aggregation Strategy:   {det_cfg['aggregation']}")
print(f"Bootstrap Iterations:   {det_cfg['n_bootstrap']}")
print(f"Bootstrap Threshold:    {det_cfg['threshold']:.6f} (99th percentile)")


## 5. Shift Detection Implementation
The modular detector `DriftDetector` is implemented in `src/drift_detection.py`.
Let us inspect the detector results on:
1. **Train -> Val** (Same season baseline: Sep–Oct 2018)
2. **Train -> Test** (Natural seasonal shift: Nov–Dec 2018)


In [ ]:
with open(DRIFT_RES / 'drift_summary.json') as f:
    drift_sum = json.load(f)

print("--- Natural Shift Detection Results ---")
print(f"Train -> Val (Sep-Oct):  Shift Score = {drift_sum['train_val_shift_score']:.6f} | Detected: {drift_sum['train_val_detected']}")
print(f"Train -> Test (Nov-Dec): Shift Score = {drift_sum['train_test_shift_score']:.6f} | Detected: {drift_sum['train_test_detected']}")
print(f"Detection Threshold:     {drift_sum['threshold']:.6f}")


## 6. Normal-Condition Evaluation (Condition A)
We evaluate our primary model—the **Phase 6 Self-Supervised (SSL) Pretrained Temporal Transformer**—on the original test set (Condition A).


In [ ]:
rob_df = pd.read_csv(DRIFT_RES / 'robustness_comparison.csv')
print("Model Performance: Condition A (Normal Test)")
rob_df[rob_df['Condition'] == 'Condition A (Normal Test)']


## 7. Shifted-Condition Evaluation (Condition B) & Performance Degradation
We evaluate the same trained model under **Condition B (Shifted Test)**, where additive mean shift ($k=1.0 \times \text{std}$) is applied to simulate sensor calibration drift.

We measure performance degradation without retraining.


In [ ]:
print("Robustness Comparison & Performance Degradation:")
rob_df


## 8. Sensor-Level Shift Analysis
We rank each sensor feature by its individual KS statistic to identify which channels deviate most strongly from the reference regime.


In [ ]:
pf_test = pd.read_csv(DRIFT_RES / 'per_feature_ks_test.csv')
print("Top 8 Shifted Features (Train -> Test):")
display(pf_test.head(8))

# Display per-feature KS visualization
Image.open(DRIFT_FIGS / 'per_feature_ks_score.png')


In [ ]:
# Distribution comparison for representative sensors
Image.open(DRIFT_FIGS / 'distribution_comparison.png')


## 9. Shift Severity Experiment
To quantify how performance degrades as distribution shift intensifies, we evaluate graded severity levels:
- **Level 0 (none)**: $k = 0.0$
- **Level 1 (mild)**: $k = 0.3 \times \text{std}$
- **Level 2 (moderate)**: $k = 0.6 \times \text{std}$
- **Level 3 (strong)**: $k = 1.0 \times \text{std}$


In [ ]:
sev_df = pd.read_csv(DRIFT_RES / 'severity_experiment.csv')
print("Severity Experiment Metrics Table:")
display(sev_df[['level', 'shift_k', 'shift_score', 'shift_detected', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc', 'fpr', 'fnr']])

# Plot severity degradation curve
Image.open(DRIFT_FIGS / 'severity_performance.png')


In [ ]:
# Plot shift score vs severity multiplier
Image.open(DRIFT_FIGS / 'shift_score_vs_severity.png')


## 10. Model Comparison Under Shift
We compare three distinct architectures under the same strong shift condition ($k=1.0$):
1. **SSL-pretrained Temporal Transformer** (Phase 6)
2. **Standard Temporal Transformer** (Phase 5)
3. **Bidirectional LSTM Baseline** (Phase 4)


In [ ]:
comp_df = pd.read_csv(DRIFT_RES / 'model_comparison_under_shift.csv')
display(comp_df)

# Grouped bar chart comparison
Image.open(DRIFT_FIGS / 'model_comparison_shift.png')


## 11. Error Analysis Under Shift
We inspect the error distributions (False Positives and False Negatives) between the Normal and Shifted conditions.


In [ ]:
err_df = pd.read_csv(DRIFT_RES / 'error_analysis.csv')
print(f"Sample error analysis records:")
display(err_df.head(10))

# False positive comparison
fp_norm = (sev_df.loc[sev_df['level']=='none', 'fp'].values[0])
fn_norm = (sev_df.loc[sev_df['level']=='none', 'fn'].values[0])
fp_shft = (sev_df.loc[sev_df['level']=='strong', 'fp'].values[0])
fn_shft = (sev_df.loc[sev_df['level']=='strong', 'fn'].values[0])

print(f"Normal Condition:  FP = {fp_norm:4d}, FN = {fn_norm:4d}")
print(f"Shifted Condition: FP = {fp_shft:4d}, FN = {fn_shft:4d}")


### Error Mechanism Findings:
1. **False Positive Explosion**: Under positive additive mean shift, normal baseline readings are elevated into ranges previously associated with anomalies, causing the static classification threshold (0.0297) to trigger false alarms (FPR jumps from 2.92% to 92.19%).
2. **Zero False Negatives**: Because all sensor values are elevated, actual anomalies also produce high anomaly scores, resulting in 100% recall (0 FN).
3. **Representation vs. Threshold Shift**: The SSL Transformer's ROC-AUC remains remarkably high (0.9575) and PR-AUC remains 0.8961 under strong shift. This indicates that the learned representations still separate true anomalies from background variations; the degradation in F1 is primarily caused by **threshold misalignment** due to the baseline offset.


## 12. Results: Temporal Drift Tracking
We monitor the rolling 7-day window shift score over time across the test period (Nov–Dec 2018).


In [ ]:
rolling_df = pd.read_csv(DRIFT_RES / 'rolling_drift_test.csv')
print(f"Computed {len(rolling_df)} rolling evaluation windows across test period.")

# Display temporal drift tracking
Image.open(DRIFT_FIGS / 'temporal_drift_tracking.png')


## 13. Limitations
1. **Univariate Aggregation**: The KS detector treats each sensor channel independently and averages the statistics. While effective and non-parametric, it does not explicitly capture inter-sensor cross-correlation drift or covariance structure changes.
2. **Synthetic Shift Structure**: The synthetic experiment applies uniform additive scaling ($k \times \sigma$) across sensor channels. Real-world physical faults might cause localized sensor-specific drift, intermittent calibration faults, or non-linear scaling.
3. **Static Thresholding**: The anomaly detection classification threshold was kept fixed during evaluation to strictly measure passive robustness; real deployment requires adaptive threshold recalibration upon drift detection.


## 14. Findings & Handoff to Phase 8

### Key Findings:
1. **Reliable Non-Parametric Detection**: The Kolmogorov-Smirnov test paired with bootstrap threshold derivation successfully identified both natural seasonal shift (KS = 0.0746 > 0.0091) and synthetic operational drift without requiring any test labels.
2. **Most Sensitive Sensors**: Actuator and boundary flow/pressure sensors (`PUMP_1`, `p235`, `p227`, `n1`) exhibited the greatest distributional divergence.
3. **SSL Representation Superiority**: The SSL-pretrained Transformer showed substantially greater representational stability under distribution shift (ROC-AUC = 0.9575) compared to the Standard Transformer (ROC-AUC = 0.2070) and LSTM (F1 = 0.0000).
4. **Primary Failure Mode**: Distribution shift causes severe Precision and F1 degradation via False Alarm explosion ($FPR > 90\%$), while Recall remains high.

### Handoff to Phase 8 (Adaptive Learning):
Because the SSL Transformer maintains high discriminative separation (high ROC-AUC) even under shifted conditions, the model does *not* necessarily require full retraining from scratch upon detecting shift. Instead, Phase 8 can explore **controlled adaptation**, including:
- Dynamic threshold adjustment guided by drift score.
- Unsupervised or few-shot adaptation / fine-tuning on incoming normal data buffers.
- Model parameter updating when shift exceeds critical operational limits.
